# 7-3. 전체 미세 조정 (Full Fine-Tuning)

「전체」는 **모든 파라미터를 건드린다**는 뜻이지, 크게 바꾼다는 뜻이 아닙니다.
파라미터를 하나도 남기지 않고 전부, 그러나 **아주 조금씩만** 업데이트합니다.
(챕터7 퀴즈 18번)

## 이 장의 결론을 먼저

> **형식은 미세 조정으로, 능력은 사전훈련으로.**

- **바뀌는 것** = 대답하는 **형식**. 묻는 말에 답하고, 다 쓰면 멈추고, 시킨 일의 모양을 갖추는 것.
- **바뀌지 않는 것** = 대답에 담기는 **내용**. 지식·추론·계산은 사전훈련에서 이미 정해졌습니다.

그래서 지시 데이터를 아무리 늘려도 **모르는 것을 알게 만들 수는 없습니다.**
같은 데이터·같은 설정으로 훈련해도 베이스 모델이 좋으면 결과가 좋은 이유가 이것입니다.

---

## 이 노트북의 구성

| 절 | 내용 | 어디서 |
|---|---|---|
| 1 | VRAM 계산 — 내 GPU로 가능한가 | 어디서나 |
| 2 | 파이프라인 전체를 **초소형 모델**로 검증 | 맥/CPU 2분 |
| 3 | 실제 베이스 모델로 SFT | 24GB GPU 또는 RunPod |
| 4 | 생성 전략 — 온도·반복 페널티 | 어디서나 |
| 5 | 외운 지식 vs 넣어준 지식 (RAG의 입구) | 어디서나 |

> 2절까지는 로컬에서 그대로 돌아갑니다. 3절은 `MODEL_ID` 만 바꾸면 같은 코드입니다.

In [ ]:
import sys, os, math, time, json
sys.path.insert(0, os.path.abspath("../.."))
sys.path.insert(0, os.path.abspath(".."))

import torch, torch.nn.functional as F
from common import get_device, set_seed, human
from chat import (render_chatml, build_sft_example, collate, load_tokenizer,
                  ByteTokenizer, IGNORE_INDEX, IM_END)

set_seed(1234)
device = get_device()

HAS_HF = True
try:
    import transformers
    print("transformers", transformers.__version__)
except ImportError:
    HAS_HF = False
    print("transformers 미설치 — 3절은 건너뜁니다.")
    print("설치:  ../../.venv311/bin/pip install -r ../requirements.txt")

## 1. 먼저 계산부터 — 내 GPU에 들어가는가

전체 미세 조정에서 VRAM을 먹는 것은 넷입니다.

| 항목 | bf16 기준 크기 | 7B 모델이면 |
|---|---|---|
| 모델 가중치 | 2 bytes × P | 14 GB |
| 그래디언트 | 2 bytes × P | 14 GB |
| 옵티마이저 상태 (AdamW: m, v) | 8 bytes × P (fp32) | 56 GB |
| 활성값 | 배치·길이에 비례 | 수 GB |

**합이 84GB+.** 4090(24GB)으로는 7B 전체 미세 조정이 불가능합니다.
이게 챕터 8에서 **PEFT/LoRA** 를 배우는 이유입니다 — 2·3번을 거의 0으로 만듭니다.

In [ ]:
def vram_estimate(n_params, optimizer="adamw", dtype_bytes=2,
                  act_gb=2.0, grad_ckpt=False):
    w  = n_params * dtype_bytes / 1e9
    g  = n_params * dtype_bytes / 1e9
    o  = n_params * (8 if optimizer == "adamw" else 4) / 1e9   # fp32 m,v
    a  = act_gb * (0.3 if grad_ckpt else 1.0)
    return dict(weights=w, grads=g, optim=o, acts=a, total=w + g + o + a)

print(f"{'모델':<22}{'가중치':>8}{'그래디언트':>11}{'옵티마이저':>11}{'활성값':>8}{'합계':>9}")
print("-" * 70)
for name, p in [("Qwen3-0.6B", 0.6e9), ("kanana-2-1.3b", 1.3e9),
                ("Qwen3-4B", 4e9), ("Llama-3-8B", 8e9)]:
    e = vram_estimate(p)
    print(f"{name:<22}{e['weights']:7.1f}G{e['grads']:10.1f}G"
          f"{e['optim']:10.1f}G{e['acts']:7.1f}G{e['total']:8.1f}G")

print("\n24GB(3090/4090) 한 장으로 전체 미세 조정이 가능한 선:")
for name, p in [("Qwen3-0.6B", 0.6e9), ("kanana-2-1.3b", 1.3e9), ("Qwen3-4B", 4e9)]:
    t = vram_estimate(p, grad_ckpt=True)["total"]
    print(f"  {name:<16} {t:5.1f}GB  -> {'가능' if t < 22 else '불가 (LoRA 필요)'}")

### 터졌을 때 쓰는 카드

| 방법 | 줄어드는 것 | 대가 |
|---|---|---|
| 배치 크기 ↓ | 활성값 | 학습 불안정 → gradient accumulation 으로 보완 |
| **Gradient Accumulation** | 활성값 | 속도 ↓ (유효 배치는 유지) |
| **Gradient Checkpointing** | 활성값 70% | 속도 20~30% ↓ |
| 시퀀스 길이 ↓ | 활성값 | 긴 대화 학습 불가 |
| 8-bit optimizer | 옵티마이저 상태 절반 | 미세한 정밀도 손실 |
| **LoRA / QLoRA** | 그래디언트 + 옵티마이저 거의 전부 | 챕터 8 |

In [ ]:
def gradient_accumulation_demo(micro_bs=2, accum=4):
    """작은 배치 여러 번의 그래디언트를 누적해 큰 배치 하나처럼 만듭니다."""
    print(f"micro batch {micro_bs} × accumulation {accum} = 유효 배치 {micro_bs*accum}")
    print("""
    for step in range(total_steps):
        for i in range(accum):
            loss = model(next(micro_batch)).loss / accum   # <- accum 으로 나누는 것이 핵심
            loss.backward()                                #    (안 나누면 LR 이 accum 배가 된 셈)
        optimizer.step(); optimizer.zero_grad()
    """)
gradient_accumulation_demo()

## 2. 파이프라인 전체를 초소형 모델로 검증

**3절에서 쓸 코드와 완전히 같은 코드**를, 랜덤 초기화한 아주 작은 Qwen3 로 돌립니다.
목적은 성능이 아니라 **배선 확인**입니다 — 토큰 길이, 마스킹, 패딩,
손실이 내려가는지, 생성이 멈추는지.

> 큰 모델에서 바로 시작하면 한 번 돌리는 데 수십 분이 걸려
> 오타 하나 찾는 데도 그만큼이 듭니다. **작게 먼저 돌리는 것이 실무 순서입니다.**

In [ ]:
TRAIN_DATA = [
    [{"role": "system", "content": "당신은 짧고 정확하게 답하는 비서입니다."},
     {"role": "user", "content": "대한민국의 수도는?"},
     {"role": "assistant", "content": "서울입니다."}],
    [{"role": "system", "content": "당신은 짧고 정확하게 답하는 비서입니다."},
     {"role": "user", "content": "물의 끓는점은?"},
     {"role": "assistant", "content": "1기압에서 섭씨 100도입니다."}],
    [{"role": "user", "content": "고맙습니다"},
     {"role": "assistant", "content": "천만에요. 더 궁금한 점 있으면 말씀해 주세요."}],
    [{"role": "user", "content": "3 더하기 4는?"},
     {"role": "assistant", "content": "7입니다."}],
    [{"role": "user", "content": "안녕하세요"},
     {"role": "assistant", "content": "안녕하세요! 무엇을 도와드릴까요?"}],
    [{"role": "user", "content": "파이썬이 뭐야?"},
     {"role": "assistant", "content": "배우기 쉬운 프로그래밍 언어입니다."}],
]
print(f"훈련 샘플 {len(TRAIN_DATA)}건 (배선 확인용 장난감 데이터)")

In [ ]:
def make_tiny_model(vocab_size):
    """랜덤 초기화한 초소형 Qwen3. transformers 가 없으면 직접 만든 TinyLM 을 씁니다."""
    if HAS_HF:
        from transformers import AutoConfig, AutoModelForCausalLM
        cfg = AutoConfig.for_model(
            "qwen3", vocab_size=vocab_size, hidden_size=128, num_hidden_layers=4,
            num_attention_heads=4, num_key_value_heads=2, intermediate_size=256,
            max_position_embeddings=512, head_dim=32, tie_word_embeddings=True)
        return AutoModelForCausalLM.from_config(cfg)

    class TinyLM(torch.nn.Module):
        def __init__(self, vocab, d=128, nhead=4, nlayer=4, maxlen=512):
            super().__init__()
            self.emb = torch.nn.Embedding(vocab, d)
            self.pos = torch.nn.Embedding(maxlen, d)
            lyr = torch.nn.TransformerEncoderLayer(d, nhead, d*4, batch_first=True,
                                                   norm_first=True, dropout=0.0)
            self.body = torch.nn.TransformerEncoder(lyr, nlayer)
            self.head = torch.nn.Linear(d, vocab)
        def forward(self, input_ids, attention_mask=None, labels=None):
            T = input_ids.size(1)
            h = self.emb(input_ids) + self.pos(torch.arange(T, device=input_ids.device))
            m = torch.triu(torch.ones(T, T, dtype=torch.bool, device=input_ids.device), 1)
            logits = self.head(self.body(h, mask=m))
            out = type("O", (), {})(); out.logits = logits; out.loss = None
            if labels is not None:
                out.loss = F.cross_entropy(logits[:, :-1].reshape(-1, logits.size(-1)),
                                           labels[:, 1:].reshape(-1),
                                           ignore_index=IGNORE_INDEX)
            return out
    return TinyLM(vocab_size)

tiny_tok = ByteTokenizer()
tiny = make_tiny_model(tiny_tok.vocab_size).to(device)
print(f"초소형 모델 파라미터: {human(sum(p.numel() for p in tiny.parameters()))}")

In [ ]:
def sft_train(model, tokenizer, data, *, epochs=40, lr=3e-4, micro_bs=3,
              accum=2, max_len=256, log_every=10):
    """7-2에서 만든 조각들을 그대로 써서 돌리는 SFT 루프.

    3절의 진짜 모델에도 **이 함수를 그대로** 씁니다.
    """
    pad_id = getattr(tokenizer, "pad_token_id", 0) or 0
    exs = [build_sft_example(tokenizer, d, mask_prompt=True) for d in data]
    exs = [e for e in exs if len(e) <= max_len]

    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0, betas=(0.9, 0.95))
    total = epochs * max(1, len(exs) // (micro_bs * accum))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, s / max(1, int(0.1 * total))) *          # 웜업 10%
                       (0.5 * (1 + math.cos(math.pi * min(1.0, s / total)))))  # 코사인 감쇠

    model.train(); hist = []; step = 0; t0 = time.time()
    for ep in range(epochs):
        for i in range(0, len(exs), micro_bs * accum):
            opt.zero_grad()
            window = exs[i:i + micro_bs * accum]
            for j in range(0, len(window), micro_bs):
                b = collate(window[j:j + micro_bs], pad_id)
                b = {k: v.to(device) for k, v in b.items()}
                loss = model(**b).loss / accum
                loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); sched.step(); step += 1
            hist.append(loss.item() * accum)
        if ep % log_every == 0 or ep == epochs - 1:
            print(f"  epoch {ep:3d}  loss {hist[-1]:.4f}  lr {sched.get_last_lr()[0]:.2e}")
    print(f"  완료: {step} step, {time.time()-t0:.1f}초")
    return hist

hist = sft_train(tiny, tiny_tok, TRAIN_DATA, epochs=120, lr=3e-3, log_every=30)

In [ ]:
@torch.no_grad()
def chat(model, tokenizer, user, system=None, max_new=80,
         temperature=0.0, repetition_penalty=1.0):
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": user}]
    prompt = render_chatml(msgs, add_generation_prompt=True)
    ids = torch.tensor([tokenizer.encode(prompt, add_special_tokens=False)], device=device)
    start = ids.size(1)
    eos = getattr(tokenizer, "eos_token_id", None) or tokenizer.encode(IM_END, add_special_tokens=False)[0]

    model.eval()
    for _ in range(max_new):
        logits = model(input_ids=ids).logits[:, -1]
        if repetition_penalty != 1.0:                 # 이미 나온 토큰의 점수를 깎는다
            for t in set(ids[0, start:].tolist()):
                logits[0, t] /= repetition_penalty
        if temperature <= 0:
            nxt = logits.argmax(-1, keepdim=True)     # greedy
        else:
            nxt = torch.multinomial(F.softmax(logits / temperature, -1), 1)
        ids = torch.cat([ids, nxt], 1)
        if nxt.item() == eos:
            break
    new = ids[0, start:].tolist()
    stopped = eos in new
    return tokenizer.decode(new).replace(IM_END, "").strip(), len(new), stopped

for q in ["대한민국의 수도는?", "3 더하기 4는?", "고맙습니다"]:
    ans, n, stopped = chat(tiny, tiny_tok, q, system="당신은 짧고 정확하게 답하는 비서입니다.")
    print(f"Q: {q}\nA: {ans}   ({n}토큰, 멈춤={stopped})\n")

> 외운 것을 그대로 뱉고 있을 뿐입니다 — 6건으로 훈련했으니 당연합니다.
> **여기서 확인한 것은 성능이 아니라 배선입니다**: 손실이 내려갔고,
> 모델이 형식을 갖춰 답했고, **스스로 멈췄습니다.**
>
> 이 세 가지가 확인되면 같은 코드를 큰 모델에 그대로 올립니다.

## 3. 실제 베이스 모델로 (24GB GPU 또는 RunPod)

`MODEL_ID` 만 바꾸면 2절의 `sft_train` / `chat` 을 그대로 씁니다.

```python
MODEL_ID = "Qwen/Qwen3-0.6B-Base"        # 또는 "kakaocorp/kanana-2-1.3b-base"
```

**Base 모델을 쓰세요.** `-Instruct` 는 이미 지시 튜닝이 끝난 모델이라
우리가 할 일이 이미 되어 있습니다.

> ⚠️ 맥(MPS)에서도 0.6B 는 돌아가지만 느립니다. 실제 훈련은
> `0704_SFT_RunPod.md` 를 따라 클라우드에서 돌리는 쪽을 권합니다.

In [ ]:
MODEL_ID = "Qwen/Qwen3-0.6B-Base"
RUN_REAL = False          # <- True 로 바꾸면 실제 모델을 받아 훈련합니다 (수 GB 다운로드)

if HAS_HF and RUN_REAL:
    from transformers import AutoModelForCausalLM

    tokenizer = load_tokenizer(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.bfloat16).to(device)
    model.gradient_checkpointing_enable()          # VRAM 절약
    model.config.use_cache = False                 # checkpointing 과 함께 쓰려면 꺼야 함

    print(f"{MODEL_ID}: {human(sum(p.numel() for p in model.parameters()))} 파라미터")

    print("\n[훈련 전]")
    for q in ["대한민국의 수도는?", "파이썬이 뭐야?"]:
        ans, n, stopped = chat(model, tokenizer, q, max_new=120)
        print(f"Q: {q}\nA: {ans[:200]}\n   ({n}토큰, 멈춤={stopped})\n")
    # -> 베이스 모델은 멈추지 않습니다. 종료 토큰을 배운 적이 없기 때문입니다.

    sft_train(model, tokenizer, TRAIN_DATA, epochs=3, lr=2e-5, micro_bs=1, accum=8)

    print("\n[훈련 후]")
    for q in ["대한민국의 수도는?", "파이썬이 뭐야?"]:
        ans, n, stopped = chat(model, tokenizer, q, max_new=120)
        print(f"Q: {q}\nA: {ans[:200]}\n   ({n}토큰, 멈춤={stopped})\n")
else:
    print("RUN_REAL = False — 건너뜁니다.")
    print("""
실제 설정값 가늠 (0.6B~1.3B, 지시 데이터 수만 건 기준)
    learning_rate   1e-5 ~ 2e-5      <- 크게 잡으면 사전훈련을 잊습니다
    epochs          2 ~ 3            <- 더 돌리면 데이터를 외웁니다
    warmup_ratio    0.03
    scheduler       cosine
    max_length      1024 ~ 2048
    effective_bs    64 ~ 128         (micro_bs × accum × GPU수)
    dtype           bfloat16
""")

### 학습률을 크게 잡으면 안 되는 이유

지시 데이터는 사전훈련에 비하면 **양도 적고 다루는 범위도 좁습니다.**
여기에 큰 학습률을 쓰면 형식은 빨리 익히는 대신
**사전훈련에서 배운 것을 잊어버립니다** — 파국적 망각(catastrophic forgetting).

### 작은 모델이 더 오래 걸릴 수도 있다

같은 데이터·같은 설정으로 세 모델을 훈련했을 때 **가장 작은 모델의 훈련 시간이
가장 길 수 있습니다.** 토크나이저가 한국어를 더 잘게 쪼개면 토큰 수 자체가 늘기 때문입니다.
(챕터7 퀴즈 23번)

In [ ]:
# 토크나이저가 한국어를 얼마나 잘게 쪼개는지 — 모델 선택의 숨은 변수
text = "전체 미세 조정으로 바뀌는 것은 대답하는 형식입니다. 알고 있는 지식은 사전훈련에서 이미 정해져 있습니다."
print(f"원문 {len(text)}글자\n")
if HAS_HF:
    for mid in ["Qwen/Qwen3-0.6B-Base", "kakaocorp/kanana-2-1.3b-base"]:
        try:
            t = load_tokenizer(mid, quiet=True)
            n = len(t.encode(text, add_special_tokens=False))
            print(f"  {mid:<34} {n:4d} 토큰  (글자당 {n/len(text):.2f})")
        except Exception as e:
            print(f"  {mid:<34} 불러오기 실패 ({type(e).__name__})")
else:
    print("  transformers 필요 — 설치 후 실행하세요.")
print("\n토큰이 많다 = 같은 데이터를 훈련하는 데 더 많은 step = 더 오래 걸린다")

## 4. 생성 전략 — 온도는 품질 손잡이가 아니다

훈련이 끝난 모델은 그대로 두고 **고르는 방법만** 바꾸는 것이 생성 전략입니다.

- **온도(temperature)** 는 **무작위성** 손잡이입니다. 올리면 확률이 낮은 토큰도 가끔 고릅니다.
  되풀이 고리에서 빠져나오는 것은 그 덕분이지, **답이 더 정확해져서가 아닙니다.**
- 너무 올리면 엉뚱한 토큰을 골라 문장이 흐트러지거나 끝맺지 못합니다.
- **반복 페널티(repetition penalty)** 는 이미 나온 토큰의 점수를 직접 깎습니다 (CTRL, 2019).

In [ ]:
# 온도가 실제로 바꾸는 것은 **다음 토큰 확률분포의 뾰족함**입니다.
# 생성 문장을 보기 전에, 분포 자체를 먼저 재 봅니다.
msgs = [{"role": "user", "content": "안녕하세요"}]
prompt = render_chatml(msgs, add_generation_prompt=True)
ids = torch.tensor([tiny_tok.encode(prompt)], device=device)

tiny.eval()
with torch.no_grad():
    logits = tiny(input_ids=ids).logits[0, -1].float().cpu()

print(f"{'T':>6}{'엔트로피':>12}{'1위 확률':>12}   상위 3개 확률")
for T in [0.3, 0.7, 1.0, 1.5, 3.0]:
    p = F.softmax(logits / T, dim=-1)
    ent = -(p * p.clamp_min(1e-12).log()).sum().item()
    top = p.topk(3).values.tolist()
    print(f"{T:6.1f}{ent:12.3f}{top[0]:12.4f}   " + " ".join(f"{x:.3f}" for x in top))

print("\n온도를 올리면 엔트로피가 커집니다 = 1위 말고 다른 토큰도 뽑힐 여지가 생깁니다.")
print("바뀐 것은 '고르는 방법'뿐입니다 — 모델 가중치도, 로짓도 그대로입니다.")

In [ ]:
# 그러면 실제 생성은 얼마나 달라질까요? 같은 질문을 여러 번 생성해 봅니다.
print("같은 질문 10회 생성 — 서로 다른 답이 몇 가지 나오는가\n")
for label, kw in [
    ("greedy (T=0)       ", dict(temperature=0.0)),
    ("T=0.7              ", dict(temperature=0.7)),
    ("T=1.5              ", dict(temperature=1.5)),
    ("T=3.0 (너무 높음)   ", dict(temperature=3.0)),
]:
    set_seed(42)
    outs = [chat(tiny, tiny_tok, "안녕하세요", max_new=60, **kw) for _ in range(10)]
    uniq = len({o[0] for o in outs})
    stop_rate = sum(o[2] for o in outs) / len(outs)
    print(f"{label} 서로 다른 답 {uniq:2d}/10   스스로 멈춘 비율 {stop_rate:.0%}")
    print(f"{'':21}예: {outs[0][0][:60]!r}")

print("""
읽는 법:
  · T 를 올릴수록 답이 다양해집니다 (되풀이 고리에서 빠져나오는 힘).
  · 너무 올리면 멈추는 비율이 떨어집니다 — 종료 토큰 대신 엉뚱한 토큰을 고르기 때문입니다.
  · 어느 쪽도 '내용이 정확해지는' 방향은 아닙니다.

주의: 이 장난감 모델은 6건에 과적합해서 분포가 매우 뾰족합니다.
     그래서 T=0.7 정도로는 거의 변화가 없고, 아주 높여야 흔들립니다.
     제대로 훈련한 모델에서는 T=0.7~1.0 구간에서 이 차이가 또렷하게 보입니다.""")

## 5. 외운 지식과, 넣어준 지식

모델이 파라미터에 담아 둘 수 있는 지식의 양은 제한돼 있고,
**작은 모델일수록 더 그렇습니다.**

근거가 될 글을 **시스템 메시지에 넣어주면** 같은 모델이 제대로 답합니다.
지식을 기억에서 꺼내는 대신 **눈앞에서 읽기** 때문입니다.

질문과 관련된 글을 찾아서 넣어주는 방식을 **RAG** (Retrieval-Augmented Generation)
라고 부릅니다. 모델을 다시 훈련하지 않고도 새 자료를 다룰 수 있습니다.
강의에서 쓴 [heegyu/korquad-chat-v1](https://huggingface.co/datasets/heegyu/korquad-chat-v1)
은 위키 지문을 시스템 메시지에 넣어 둔 대화 데이터입니다.

In [ ]:
PASSAGE = ("경복궁은 1395년 조선 태조 때 지어진 조선의 법궁이다. "
           "임진왜란 때 불탔다가 1867년 흥선대원군 주도로 중건되었다.")
QUESTION = "경복궁은 언제 중건되었나요?"

print("[근거 글 없이]")
print("  system: (없음)")
print(f"  user  : {QUESTION}")
print("  -> 작은 모델은 기억에 없는 것을 지어내거나, 되풀이하며 상한까지 이어 씁니다.\n")

print("[근거 글을 시스템 메시지에 넣고]")
print(f"  system: {PASSAGE}")
print(f"  user  : {QUESTION}")
print("  -> 그 글 안의 말로 짧게 답하고 멈춥니다. (챕터7 퀴즈 24번)\n")

msgs = [{"role": "system", "content": PASSAGE}, {"role": "user", "content": QUESTION}]
prompt = render_chatml(msgs, add_generation_prompt=True)
print(f"실제로 모델에 들어가는 토큰 수: {len(tiny_tok.encode(prompt))}")
print("근거 글이 길수록 컨텍스트를 많이 먹습니다 — RAG 에서 '무엇을 넣을까'가 핵심인 이유입니다.")
print("\n(RUN_REAL=True 로 실제 모델을 띄우면 두 경우의 차이를 직접 볼 수 있습니다.)")

## 정리

| 질문 | 답 |
|---|---|
| 미세 조정이 바꾸는 것은? | **형식**. 내용은 사전훈련에서 결정됨 |
| 지시 데이터를 늘리면 지식이 늘까? | 아니오. 필요한 지식은 **넣어주는**(RAG) 쪽이 현실적 |
| 학습률을 크게 잡으면? | 형식은 빨리 익히고 **사전훈련을 잊음** |
| 온도를 올리면? | 다양해질 뿐, **정확해지지 않음** |
| 가린 토큰을 입력에서도 뺄까? | 아니오. **손실에서만** 뺌 |
| 종료 토큰을 가릴까? | 아니오. 가리면 **멈추는 법을 못 배움** |

### 챕터 7에서 챕터 8로

전체 미세 조정은 **된다는 것은 확인했지만 비쌉니다.** 7B면 80GB+.
그래서 다음이 **PEFT/LoRA** 입니다 — 원본 가중치는 얼리고,
끼워 넣은 작은 행렬에만 그래디언트를 계산해 **개인 GPU로 끌어내립니다.**

그리고 지금까지 만든 것은 「형식을 아는 비서」일 뿐입니다.
**특정한 인격과 말투**를 입히는 것이 챕터 8의 페르소나 튜닝이고,
그 데이터를 사람이 다 쓸 수 없으니 **합성 데이터**를 만듭니다.

> **다음** → `0704_SFT_RunPod.md` (실제 GPU에서 돌리기)
> → `../Ch8/` (챕터 8 — 아직 미공개)